In [1]:
!pip install pandas

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [1]:
import os
import pandas as pd
import numpy as np
#import networkx as nx
#import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [3]:
hidden_units = [32, 32]
learning_rate = 0.01
dropout_rate = 0.5
num_epochs = 2
batch_size = 8 #256


In [4]:
def compile_model(model):
    # Compile the model.
    model.compile(
        optimizer="rmsprop",
        #optimizer=keras.optimizers.Adam(learning_rate),
        loss="binary_crossentropy",
        # Tati: categorical_crossentropy, expects the labels to follow a categorical encoding. 
        #       With integer labels, you should use sparse_categorical_crossentropy.
        #       This new loss function is still mathematically the same as categorical_crossentropy; it just has a different interface.
        #metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )
    # Create an early stopping callback.
    #early_stopping = keras.callbacks.EarlyStopping(
    #    monitor="val_acc", patience=50, restore_best_weights=True
    #
    #)
    return model

# This function trains an input model using the given training data.
def run_experiment(model, x_train, y_train): 
    guardarModelo = keras.callbacks.ModelCheckpoint(
        #filepath="/mnt/modelos1/",  # "checkpoint_path.keras",
        filepath="/mnt/modelos_iscx/",
        monitor="val_loss",
        save_best_only=True,
        save_format="tf",
    )
    # Fit the model.
    history = model.fit(
        x=x_train,
        y=y_train,
        epochs=num_epochs,
        batch_size=batch_size,
        validation_split=0.15,
        callbacks=[guardarModelo], #early_stopping],
    )

    return history


def create_ffn(hidden_units, dropout_rate, name=None):
    fnn_layers = []

    for units in hidden_units:
        fnn_layers.append(layers.BatchNormalization())
        fnn_layers.append(layers.Dropout(dropout_rate))
        fnn_layers.append(layers.Dense(units, activation=tf.nn.gelu))

    return keras.Sequential(fnn_layers, name=name)


In [5]:
class GraphConvLayer(layers.Layer):
    def __init__(
        self,
        hidden_units,
        dropout_rate=0.2,
        aggregation_type="mean",
        combination_type="concat",
        normalize=False,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.aggregation_type = aggregation_type
        self.combination_type = combination_type
        self.normalize = normalize

        self.ffn_prepare = create_ffn(hidden_units, dropout_rate)
        if self.combination_type == "gated":
            self.update_fn = layers.GRU(
                units=hidden_units,
                activation="tanh",
                recurrent_activation="sigmoid",
                dropout=dropout_rate,
                return_state=True,
                recurrent_dropout=dropout_rate,
            )
        else:
            self.update_fn = create_ffn(hidden_units, dropout_rate)

    def prepare(self, node_repesentations, weights=None):
        # node_repesentations shape is [num_edges, embedding_dim].
        messages = self.ffn_prepare(node_repesentations)
        if weights is not None:
            messages = messages * tf.expand_dims(weights, -1)
        return messages

    def aggregate(self, node_indices, neighbour_messages, node_repesentations):
        # node_indices shape is [num_edges].
        # neighbour_messages shape: [num_edges, representation_dim].
        # node_repesentations shape is [num_nodes, representation_dim]
        num_nodes = node_repesentations.shape[0]
        if self.aggregation_type == "sum":
            aggregated_message = tf.math.unsorted_segment_sum(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        elif self.aggregation_type == "mean":
            aggregated_message = tf.math.unsorted_segment_mean(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        elif self.aggregation_type == "max":
            aggregated_message = tf.math.unsorted_segment_max(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        else:
            raise ValueError(f"Invalid aggregation type: {self.aggregation_type}.")

        return aggregated_message

    def update(self, node_repesentations, aggregated_messages):
        # node_repesentations shape is [num_nodes, representation_dim].
        # aggregated_messages shape is [num_nodes, representation_dim].
        if self.combination_type == "gru":
            # Create a sequence of two elements for the GRU layer.
            h = tf.stack([node_repesentations, aggregated_messages], axis=1)
        elif self.combination_type == "concat":
            # Concatenate the node_repesentations and aggregated_messages.
            h = tf.concat([node_repesentations, aggregated_messages], axis=1)
        elif self.combination_type == "add":
            # Add node_repesentations and aggregated_messages.
            h = node_repesentations + aggregated_messages
        else:
            raise ValueError(f"Invalid combination type: {self.combination_type}.")

        # Apply the processing function.
        node_embeddings = self.update_fn(h)
        if self.combination_type == "gru":
            node_embeddings = tf.unstack(node_embeddings, axis=1)[-1]

        if self.normalize:
            node_embeddings = tf.nn.l2_normalize(node_embeddings, axis=-1)
        return node_embeddings

    def call(self, inputs):
        """Process the inputs to produce the node_embeddings.

        inputs: a tuple of three elements: node_repesentations, edges, edge_weights.
        Returns: node_embeddings of shape [num_nodes, representation_dim].
        """

        node_repesentations, edges, edge_weights = inputs
        # Get node_indices (source) and neighbour_indices (target) from edges.
        node_indices, neighbour_indices = edges[0], edges[1]
        # neighbour_repesentations shape is [num_edges, representation_dim].
        neighbour_repesentations = tf.gather(node_repesentations, neighbour_indices)

        # Prepare the messages of the neighbours.
        neighbour_messages = self.prepare(neighbour_repesentations, edge_weights)
        # Aggregate the neighbour messages.
        aggregated_messages = self.aggregate(
            node_indices, neighbour_messages, node_repesentations
        )
        # Update the node embedding with the neighbour messages.
        return self.update(node_repesentations, aggregated_messages)


In [6]:
class GNNNodeClassifier(tf.keras.Model):
    def __init__(
        self,
        graph_info,
        num_classes,
        hidden_units,
        aggregation_type="sum",
        combination_type="concat",
        dropout_rate=0.2,
        normalize=True,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        # Unpack graph_info to three elements: node_features, edges, and edge_weight.
        node_features, edges, edge_weights = graph_info
        self.node_features = node_features
        self.edges = edges
        self.edge_weights = edge_weights
        # Set edge_weights to ones if not provided.
        if self.edge_weights is None:
            self.edge_weights = tf.ones(shape=edges.shape[1])
        # Scale edge_weights to sum to 1.
        self.edge_weights = self.edge_weights / tf.math.reduce_sum(self.edge_weights)

        # Create a process layer.
        self.preprocess = create_ffn(hidden_units, dropout_rate, name="preprocess")
        # Create the first GraphConv layer.
        self.conv1 = GraphConvLayer(
            hidden_units,
            dropout_rate,
            aggregation_type,
            combination_type,
            normalize,
            name="graph_conv1",
        )
        # Create the second GraphConv layer.
        self.conv2 = GraphConvLayer(
            hidden_units,
            dropout_rate,
            aggregation_type,
            combination_type,
            normalize,
            name="graph_conv2",
        )
        # Create a postprocess layer.
        self.postprocess = create_ffn(hidden_units, dropout_rate, name="postprocess")
        # Create a compute logits layer.
        self.compute_logits = layers.Dense(units=num_classes, name="logits")  
          # Tati: For Tensorflow: logits is a name that it is thought to imply that this Tensor is the quantity that is being mapped to probabilities by the Softmax

    def call(self, input_node_indices):
        # Preprocess the node_features to produce node representations.
        x = self.preprocess(self.node_features)
        # Apply the first graph conv layer.
        x1 = self.conv1((x, self.edges, self.edge_weights))
        # Skip connection.
        x = x1 + x
        # Apply the second graph conv layer.
        x2 = self.conv2((x, self.edges, self.edge_weights))
        # Skip connection.
        x = x2 + x
        # Postprocess node embedding.
        x = self.postprocess(x)
        # Fetch node embeddings for the input node_indices.
        node_embeddings = tf.gather(x, input_node_indices)
        # Compute logits
        return self.compute_logits(node_embeddings)


In [7]:
# Training
training_grafos = pd.read_csv(
    "/mnt/training_GRAFOS.pkts.ncol",
    sep=" ",  
    header=None,  # no heading row
    names=["source", "target", "weight"],  # set our own names for the columns
)

training_features_tmp = pd.read_csv(
    "/mnt/training_FEATURES.pkts.SINnorm.csv",
    sep=",",  
    header=0
)


# Validation
validation_grafos = pd.read_csv(
    "/mnt/validation_GRAFOS.pkts.ncol",
    sep=" ",  
    header=None,  # no heading row
    names=["source", "target", "weight"],  # set our own names for the columns
)

validation_features_tmp = pd.read_csv(
    "/mnt/validation_FEATURES.pkts.SINnorm.csv",
    sep=",",  
    header=0
)



class_values = sorted(training_features_tmp["label"].unique())
class_idx = {name: id for id, name in enumerate(class_values)}
node_idx_train = {name: idx for idx, name in enumerate(sorted(training_features_tmp["node"].unique()))}
node_idx_val = {name: idx for idx, name in enumerate(sorted(validation_features_tmp["node"].unique()))}

training_grafos["source"] = training_grafos["source"].apply(lambda name: node_idx_train[name])
training_grafos["target"] = training_grafos["target"].apply(lambda name: node_idx_train[name])

training_features = training_features_tmp.loc[:,["node","ID","OD","IDW","ODW","label"]].copy()
training_features["node"] = training_features_tmp["node"].apply(lambda name: node_idx_train[name])
training_features["label"] = training_features_tmp["label"].apply(lambda value: class_idx[value])

validation_grafos["source"] = validation_grafos["source"].apply(lambda name: node_idx_val[name])
validation_grafos["target"] = validation_grafos["target"].apply(lambda name: node_idx_val[name])

validation_features = validation_features_tmp.loc[:,["node","ID","OD","IDW","ODW","label"]].copy()
validation_features["node"] = validation_features_tmp["node"].apply(lambda name: node_idx_val[name])
validation_features["label"] = validation_features_tmp["label"].apply(lambda value: class_idx[value])


# Create an edges array (adjacency matrix) of shape [2, num_edges]
training_edges = training_grafos[["source", "target"]].to_numpy().T
validation_edges = validation_grafos[["source", "target"]].to_numpy().T

# Create an edge weights array.
training_edge_weights = training_grafos[["weight"]].to_numpy().T 
training_edge_weights = training_edge_weights.reshape((training_edges.shape[-1],))
training_edge_weights = tf.convert_to_tensor(training_edge_weights)

# Create a node features array of shape [num_nodes, num_features].
feature_names = set(training_features.columns) - {"node", "label"}
training_node_features = tf.cast(
    training_features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32
)

# Create graph info tuple with node_features, edges, and edge_weights.
graph_info = (training_node_features, training_edges, training_edge_weights)

print("Edges shape:", training_edges.shape)
print("Nodes shape:", training_node_features.shape)


<ipython-input-7-d4b50cdc2e0e>:64: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  training_features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32


Edges shape: (2, 5020771)
Nodes shape: (2398933, 4)


In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((images, labels))

In [8]:
## GNN
gnn_model = GNNNodeClassifier(
    graph_info=graph_info,
    num_classes=2,
    hidden_units=hidden_units,
    dropout_rate=dropout_rate,
    name="gnn_model",
)

print("GNN output shape:", gnn_model([1, 10, 100]))

print(gnn_model.summary())

modelo = compile_model(gnn_model)

#train_data = ambos_features.iloc[:37943,:]
#test_data = ambos_features.iloc[37943:,:]

train_data = training_features.sample(frac=1)
test_data = validation_features.sample(frac=1)

print("Train data shape:", train_data.shape) 
print("Test data shape:", test_data.shape) 

# Create train and test features as a numpy array.
x_train = train_data[feature_names].to_numpy()
x_test = test_data[feature_names].to_numpy()
# Create train and test targets as a numpy array.
y_train = train_data["label"]
y_test = test_data["label"]


x_train = train_data.node.to_numpy()
print("x_train.dtype = ", x_train.dtype)
print("y_train.dtype = ", y_train.dtype)

#history = run_experiment(modelo, x_train, y_train)


GNN output shape: tf.Tensor(
[[-0.51589584 -0.16580537]
 [ 0.15424332 -0.05034565]
 [ 0.15424332 -0.05034565]], shape=(3, 2), dtype=float32)
Model: "gnn_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 preprocess (Sequential)     (2398933, 32)             1360      
                                                                 
 graph_conv1 (GraphConvLayer  multiple                 5888      
 )                                                               
                                                                 
 graph_conv2 (GraphConvLayer  multiple                 5888      
 )                                                               
                                                                 
 postprocess (Sequential)    (2398933, 32)             2368      
                                                                 
 logits (Dense)              multiple           

<ipython-input-8-610d830bb2e9>:26: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  x_train = train_data[feature_names].to_numpy()
<ipython-input-8-610d830bb2e9>:27: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  x_test = test_data[feature_names].to_numpy()


In [13]:
training_features.head()

,node,ID,OD,IDW,ODW,label
0,0,0,1,0,2239,0
1,252661,6,0,2907,0,0
2,1,1,1,6,6,0
3,97219,525631,540374,4796296,4936679,0
4,2,1,1,2,2,0


In [14]:
x_train

array([ 790171,  580089, 2119603, ..., 1217148, 2094851,  583581])

In [15]:
train_data.head()

,node,ID,OD,IDW,ODW,label
2016504,790171,1,0,2,0,0
534827,580089,1,1,1,1,0
1636645,2119603,1,1,1,1,0
411055,443333,1,1,3,3,0
1185943,1710239,1,1,1,1,0


In [16]:
y_train

2016504    0
534827     0
1636645    0
411055     0
1185943    0
          ..
1686292    0
694410     0
663064     0
1513545    0
538007     0
Name: label, Length: 2398933, dtype: int64

In [17]:
history = run_experiment(modelo, x_train, y_train)

Epoch 1/2


ResourceExhaustedError: Graph execution error:

Detected at node 'gnn_model/graph_conv1/sequential/dense_2/MatMul' defined at (most recent call last):
    File "/usr/lib/python3.8/runpy.py", line 194, in _run_module_as_main
      return _run_code(code, main_globals, None,
    File "/usr/lib/python3.8/runpy.py", line 87, in _run_code
      exec(code, run_globals)
    File "/usr/local/lib/python3.8/dist-packages/ipykernel_launcher.py", line 16, in <module>
      app.launch_new_instance()
    File "/usr/local/lib/python3.8/dist-packages/traitlets/config/application.py", line 982, in launch_instance
      app.start()
    File "/usr/local/lib/python3.8/dist-packages/ipykernel/kernelapp.py", line 505, in start
      self.io_loop.start()
    File "/usr/local/lib/python3.8/dist-packages/tornado/platform/asyncio.py", line 215, in start
      self.asyncio_loop.run_forever()
    File "/usr/lib/python3.8/asyncio/base_events.py", line 570, in run_forever
      self._run_once()
    File "/usr/lib/py

---

# iscx2012 analysis

In [1]:
import os
os.listdir("/root/spektral/datasets/CTU13_original_durRate/")

['graph_20110816_original_durRate.npz',
 'graph_20110816-3_original_durRate.npz',
 'graph_20110811_original_durRate.npz',
 'graph_20110818_original_durRate.npz',
 'graph_20110817_original_durRate.npz',
 'graph_20110819_original_durRate.npz',
 'graph_20110812_original_durRate.npz',
 'graph_20110815-3_original_durRate.npz',
 'graph_20110815-2_original_durRate.npz',
 'graph_20110815_original_durRate.npz',
 'graph_20110810_original_durRate.npz',
 'graph_20110818-2_original_durRate.npz',
 'graph_20110816-2_original_durRate.npz']

In [1]:
from spektral.data import DisjointLoader, BatchLoader, Dataset, Graph
import numpy as np

class CTU13_original_durRate(Dataset):
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def download(self):
        os.mkdir(self.path)
        
        captures = ["10","11","12","15","15-2","16","16-2","16-3","17","18","18-2","19","15-3"]
        
        # para poder descargar los archivos ncol de cada captura y las features de cada nodo

        features_links = ["ktAgJS22kBXcMsT", 
                          "qfM8pjZgDi9EFfd",
                          "34PWCn9JmdDA9HE",
                          "YtcCmiNEyPiYB4Y",
                          "D2HdaEZYcFz5ryt",
                          "cTY262moFLGjtmn",
                          "TbfPFbSKWqYqR7n",
                          "s2GTjFz8rNxbs4z",
                          "9dZPAENNACreDEK",
                          "bwR2Zrky49JjtgA",
                          "CmYc9JyBsHwzaYD",
                          "TNSkGJcq2CPoFtM",
                          "XwZFrQYzMLNJxAY"
        ]
        
        
        dur_rate = ["nFprrYtrZsW8Fwj",
                    "WEzZ9CYcFHPkYWk",
                    "wJHqrAgwbQLrFz3",
                    "fY7KkRzr6Re9BWP",
                    "4CHHTiYWqXCs9Ln",
                    "4orMAcGx9xitKLa",
                    "Jb9Qm35tG2QagkC",
                    "ikjFiFaMLWr8scL",
                    "Y2aSjmCxKxQ6oWZ",
                    "YZFQpTTL7cxLNgY",
                    "Z34rnW6YXMSLZKm",
                    "qA4KZ25WC4BwZKn",
                    "6ZiXnr99jLxE2oC"
        ]

        
        ncol_links = ["B5EBDnAr4z55cc9",
                      "Pz4ba4jn3nCNgAp",
                      "EbkwSBHyAkHmdHE",
                      "ttyoxLc36s7ABCB",
                      "R3b9fe25x6ncoaT",
                      "wFZ72f9kL3XFki6",
                      "7EcYp9ACPqkQqDs",
                      "YcTZCARwKCY2jiB",
                      "3cc8mcTZaEC9LGM",
                      "NDgw4PwXAwQKgb2",
                      "wY38ypkj7QSJYib",
                      "dEZYJ84z53ozZZo",
                      "NKdZfBX6DG9nB8o"            
        ]
        
        for i in range(len(captures)):
            # x = nodes features (Dur, Rate)
            # a = adjacency matrix
            # y =labels
            
            # Read files with nodes features (csv file) and connections between nodes (ncol file)
            x_4label = pd.read_csv(f'https://nube.ingenieria.uncuyo.edu.ar/s/{features_links[i]}/download', sep=",", header=0)
            x_4label = x_4label.sort_values("node")
            
            x_tmp = pd.read_csv(f'https://nube.ingenieria.uncuyo.edu.ar/s/{dur_rate[i]}/download', sep=",", header=0)
    
            a_tmp = pd.read_csv(f'https://nube.ingenieria.uncuyo.edu.ar/s/{ncol_links[i]}/download', sep=" ", header=None, names=["source", "target", "weight"])
            
            
            # Create dictionaries that identify each node and label with an integer
            node_idx = {name: idx for idx, name in enumerate(sorted(x_4label["node"].unique()))}
            
            # Change node names and label for their corresponding integer
            a_tmp["source"] = a_tmp["source"].apply(lambda name: node_idx[name])
            a_tmp["target"] = a_tmp["target"].apply(lambda name: node_idx[name])
            
            # Node features: (Dur, Rate)
            x = x_tmp.sort_values("node")[x_tmp.columns.difference(["node"], sort=False)].to_numpy()       
            x = x.astype(np.float32)
            
            # Separate source, target and weight to create a sparce matrix
            a_source = a_tmp[["source"]].to_numpy().T
            a_source = np.reshape(a_source, a_source.shape[-1])
            a_target = a_tmp[["target"]].to_numpy().T
            a_target = np.reshape(a_target, a_target.shape[-1])
            a_weight = a_tmp[["weight"]].to_numpy().T
            a_weight = np.reshape(a_weight, a_weight.shape[-1])
            # Adjacency matrix:
            a = sparse.csr_matrix((a_weight, (a_source, a_target)), shape=(x.shape[0], x.shape[0]), dtype=np.float32)
            
            # Label (CTU13 original):
            class_idx = {'normal': 0, 'infected':1}
            x_4label["label"] = x_4label["label"].apply(lambda value: class_idx[value])
            
            y = x_4label.sort_values("node")["label"].to_numpy()
            y = y.astype(np.float32)
            y = _normalize(y[:, None], "ohe") #one-hot encoding

                
            # Save in format npz
            filename = os.path.join(self.path, f'graph_201108{captures[i]}_original_durRate.npz')
            np.savez(filename, x=x, a=a, y=y)

            # Free memory
            del x_4label, x_tmp, x, a_tmp, a_source, a_target, a_weight, a, y
            gc.collect()


    def read(self):
        # We must return a list of Graph objects
        output = []
        
        captures = ["10","11","12","15","15-2","16","16-2","16-3","17","18","18-2","19","15-3"]

        for i in captures:
            data = np.load(os.path.join(self.path, f'graph_201108{i}_original_durRate.npz'), allow_pickle=True)
            output.append(
                Graph(x=data['x'], a=data['a'][()], y=data['y']) # también puede ser a=data['a'].item()
            )

        return output
    


In [8]:
datos = CTU13_original_durRate()

In [11]:
datos[0].x

array([[3.1280095e+03, 1.0200886e-01],
       [6.4778000e+01, 1.6981100e-01],
       [1.0804866e+01, 2.7765301e-01],
       ...,
       [1.0087467e-01, 9.9133053e+00],
       [1.0092783e-01, 9.9081430e+00],
       [1.0111883e-01, 9.8948593e+00]], dtype=float32)

In [13]:
os.listdir("/mnt")

['ctu13', 'NetSecGame-copy', 'synthetic', '__pycache__', 'NetSecGame', 'fides']

In [2]:
import pandas as pd

In [2]:
df = pd.read_csv("/mnt/iscx2012/iscx2012.11.143.csv")
df.columns = df.columns.str.strip()

In [71]:
df_subset=df.iloc[:,[2,3,12,13,14,15,16,17,18,19,20,21,22,23,24,25,28]].copy()
df_subset.columns

Index(['sport', 'dport', 'flowprotocol', 'flowconntime', 'numpackets',
       'numbytes', 'bps', 'bpp', 'load', 'rate', 'aggsip', 'aggdip',
       'aggdipsport', 'aggsipdport', 'aggdipsamedport', 'aggnumberofflows',
       'snortclasses'],
      dtype='object')

In [77]:
src2dst = df_subset[["sport", "dport", 'flowprotocol', 'flowconntime', 'numpackets',
       'numbytes', 'bps', 'bpp', 'load', 'rate', 'aggsip', 'aggdip',
       'aggdipsport', 'aggsipdport', 'aggdipsamedport', 'aggnumberofflows',
       'snortclasses']].copy()
dst2src = df_subset[["dport", "sport", 'flowprotocol', 'flowconntime', 'numpackets',
       'numbytes', 'bps', 'bpp', 'load', 'rate', 'aggsip', 'aggdip',
       'aggdipsport', 'aggsipdport', 'aggdipsamedport', 'aggnumberofflows',
       'snortclasses']].copy()

src2dst.rename(columns={'sport': 'origin', 'dport': 'destination', 'numbytes': 'weight'}, inplace=True)
dst2src.rename(columns={'dport': 'origin', 'sport': 'destination', 'numbytes': 'weight'}, inplace=True)

concatAll = pd.concat([src2dst, dst2src], ignore_index=True)
finalDF = concatAll.groupby(['origin','destination'], as_index=False).mean().copy()
finalDF

,origin,destination,flowprotocol,flowconntime,numpackets,weight,bps,bpp,load,rate,aggsip,aggdip,aggdipsport,aggsipdport,aggdipsamedport,aggnumberofflows,snortclasses
0,0,0,2.0,0.000000,1.0,105.75,0.000000,105.750000,0.000000e+00,0.000000,5.0,4.0,2.0,1.0,4.0,13.0,1.0
1,22,4133,0.0,5.447715,72.0,10252.00,1881.889930,142.388889,1.484366e+04,13.032988,2.0,1.0,2.0,1.0,1.0,4.0,0.0
2,22,4134,0.0,1.799327,31.0,5530.00,3073.371322,178.387097,2.376889e+04,16.672900,1.0,9.0,1.0,3.0,1.0,53.0,0.0
3,22,4135,0.0,5.444115,73.0,10312.00,1894.155432,141.260274,1.494311e+04,13.225290,1.0,3.0,1.0,3.0,1.0,16.0,0.0
4,22,4136,0.0,5.315659,76.0,17504.00,3292.912506,230.315789,2.599264e+04,14.109257,2.0,2.0,1.0,1.0,2.0,13.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2629,64144,53,1.0,0.001131,3.0,446.00,394341.290893,148.666667,1.577365e+06,1768.346600,5.0,9.0,2.0,1.0,9.0,44.0,0.0
2630,65032,53,1.0,0.253372,2.0,230.00,907.756185,115.000000,0.000000e+00,3.946766,12.0,12.0,1.0,1.0,12.0,35.0,1.0
2631,65114,53,1.0,0.011820,2.0,244.00,20642.978003,122.000000,0.000000e+00,84.602371,6.0,5.0,2.0,1.0,5.0,12.0,1.0
2632,65250,53,1.0,0.011206,2.0,564.00,50330.180261,282.000000,0.000000e+00,89.237907,32.0,32.0,1.0,1.0,32.0,70.0,1.0


In [88]:
labels = finalDF[["origin","snortclasses"]].groupby(['origin'], as_index=False).mean()
labels["snortclasses"] = labels["snortclasses"].apply(lambda x: 0 if x < 0.5 else 1)
labels

,origin,snortclasses
0,0,1
1,22,0
2,25,0
3,53,1
4,80,0
...,...,...
1304,64144,0
1305,65032,1
1306,65114,1
1307,65250,1


In [90]:
adj_aux = finalDF[["origin","destination","weight"]]
adj_aux

,origin,destination,weight
0,0,0,105.75
1,22,4133,10252.00
2,22,4134,5530.00
3,22,4135,10312.00
4,22,4136,17504.00
...,...,...,...
2629,64144,53,446.00
2630,65032,53,230.00
2631,65114,53,244.00
2632,65250,53,564.00


In [91]:
x_aux = finalDF[["origin",'flowprotocol', 'flowconntime', 'numpackets',
                    'bps', 'bpp', 'load', 'rate', 'aggsip', 'aggdip',
       'aggdipsport', 'aggsipdport', 'aggdipsamedport', 'aggnumberofflows']].groupby(['origin'], as_index=False).mean()
x_aux

,origin,flowprotocol,flowconntime,numpackets,bps,bpp,load,rate,aggsip,aggdip,aggdipsport,aggsipdport,aggdipsamedport,aggnumberofflows
0,0,2.0,0.000000,1.000000,0.000000,105.750000,0.000000e+00,0.000000,5.000000,4.000000,2.000000,1.000000,4.000000,13.000000
1,22,0.0,4.840568,163.333333,21884.087157,263.284133,1.742701e+05,35.812406,2.000000,3.200000,1.400000,1.800000,1.533333,17.400000
2,25,0.0,0.169958,12.000000,5510.583923,76.666667,4.010747e+04,65.887415,6.400000,1.000000,2.000000,1.000000,1.000000,11.400000
3,53,1.0,0.319356,2.058837,26727.135774,188.858504,5.254793e+04,76.066976,9.158995,8.065883,1.194046,1.069408,7.948232,28.610217
4,80,0.0,19.870398,50.626709,14154.455425,383.143202,1.042202e+05,42.332722,13.319664,11.711356,1.282860,1.018402,11.676656,23.807045
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1304,64144,1.0,0.001131,3.000000,394341.290893,148.666667,1.577365e+06,1768.346600,5.000000,9.000000,2.000000,1.000000,9.000000,44.000000
1305,65032,1.0,0.253372,2.000000,907.756185,115.000000,0.000000e+00,3.946766,12.000000,12.000000,1.000000,1.000000,12.000000,35.000000
1306,65114,1.0,0.011820,2.000000,20642.978003,122.000000,0.000000e+00,84.602371,6.000000,5.000000,2.000000,1.000000,5.000000,12.000000
1307,65250,1.0,0.011206,2.000000,50330.180261,282.000000,0.000000e+00,89.237907,32.000000,32.000000,1.000000,1.000000,32.000000,70.000000


In [99]:
from scipy import sparse
a_source = adj_aux[["origin"]].to_numpy().T
a_source = np.reshape(a_source, a_source.shape[-1])
a_target = adj_aux[["destination"]].to_numpy().T
a_target = np.reshape(a_target, a_target.shape[-1])
a_weight = adj_aux[["weight"]].to_numpy().T
a_weight = np.reshape(a_weight, a_weight.shape[-1])
#a = sparse.csr_matrix((a_weight, (a_source, a_target)), shape=(x_aux.shape[0], x_aux.shape[0]), dtype=np.float32)
a_source

array([    0,    22,    22, ..., 65114, 65250, 65505])

In [67]:
set(df_subset["dport"])

{0,
 22,
 25,
 53,
 80,
 110,
 137,
 138,
 139,
 143,
 443,
 1090,
 1112,
 1452,
 1863,
 2354,
 2630,
 2631,
 2756,
 5353,
 50849}

In [66]:
80 in set(df_subset1["sport"])

False

In [59]:
50849 in set(df_subset1["sport"]) #22,25,53,110,139,443,1452,1863,2630,2631,2756,50849

False

In [70]:
df_subset.loc[df_subset["dport"]==22]

,sport,dport,flowprotocol,flowconntime,numpackets,numbytes,bps,bpp,load,rate,aggsip,aggdip,aggdipsport,aggsipdport,aggdipsamedport,aggnumberofflows,snortclasses,node
1001,4133,22,0,5.447715,72,10252,1881.889930,142.388889,1.484366e+04,13.032988,2,1,2,1,1,4,0,"(4133, 22)"
1054,4134,22,0,1.799327,31,5530,3073.371322,178.387097,2.376889e+04,16.672900,1,9,1,3,1,53,0,"(4134, 22)"
1279,4135,22,0,5.444115,73,10312,1894.155432,141.260274,1.494311e+04,13.225290,1,3,1,3,1,16,0,"(22, 4135)"
1291,4136,22,0,5.315659,76,17504,3292.912506,230.315789,2.599264e+04,14.109257,2,2,1,1,2,13,0,"(4136, 22)"
1306,4137,22,0,5.372204,73,15752,2932.129904,215.780822,2.313241e+04,13.402321,1,1,1,1,1,13,0,"(4137, 22)"
1322,4139,22,0,5.307108,70,10976,2068.169707,156.800000,1.630568e+04,13.001431,4,4,3,2,2,17,1,"(4139, 22)"
1336,4141,22,0,5.308474,67,9558,1800.517437,142.656716,1.418713e+04,12.432951,5,4,3,2,2,20,0,"(4141, 22)"
1352,4142,22,0,5.339936,57,7910,1481.291162,138.771930,1.163909e+04,10.487018,1,2,1,2,1,14,0,"(4142, 22)"
1363,4143,22,0,5.376897,88,33348,6202.090165,378.954545,4.904688e+04,16.180336,2,3,1,2,2,12,0,"(22, 4143)"
1384,4144,22,0,5.348510,56,7850,1467.698481,140.178571,1.152919e+04,10.283238,2,5,1,3,2,23,0,"(4144, 22)"


In [63]:
df_subset1=df_subset[df_subset['dport'].isin(set(df_subset["sport"]))]
df_subset1

,sport,dport,flowprotocol,flowconntime,numpackets,numbytes,bps,bpp,load,rate,aggsip,aggdip,aggdipsport,aggsipdport,aggdipsamedport,aggnumberofflows,snortclasses,node
0,50850,80,0,0.240693,12,4107,17063.229924,342.250000,124108.304688,45.701370,0,0,0,0,0,0,0,"(80, 50850)"
1,50851,80,0,52.825218,36,8802,166.624963,244.500000,1294.987549,0.662562,1,1,1,1,1,1,1,"(80, 50851)"
2,143,1090,0,480.036530,15,1010,2.104007,67.333333,15.632144,0.029164,1,1,1,1,1,2,0,"(1090, 143)"
3,50852,80,0,0.255126,11,3425,13424.739149,311.363636,96673.796875,39.196319,2,1,1,1,1,3,0,"(80, 50852)"
4,2497,80,0,14.436906,2,120,8.312030,60.000000,0.000000,0.069267,1,1,1,1,1,4,0,"(80, 2497)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1466,4391,80,0,0.719910,12,4645,6452.195413,387.083333,46928.085938,15.279687,11,11,1,1,11,15,0,"(80, 4391)"
1467,4392,80,0,0.721692,15,7003,9703.585463,466.866667,72086.148438,19.398856,12,12,1,1,12,16,0,"(4392, 80)"
1468,4393,80,0,0.717314,14,6017,8388.237229,429.785714,61953.343750,18.123165,11,11,1,1,11,14,0,"(80, 4393)"
1469,4394,80,0,0.713231,12,4919,6896.783791,409.916667,50160.464844,15.422773,12,12,1,1,12,15,0,"(80, 4394)"


In [15]:
set(df_subset["sport"])-set(df_subset["dport"])

{24425,
 32780,
 4127,
 4132,
 4133,
 4134,
 4135,
 4136,
 4137,
 4138,
 4139,
 4140,
 4141,
 4142,
 4143,
 4144,
 4145,
 4146,
 4147,
 4148,
 4149,
 53307,
 57414,
 4170,
 4178,
 4179,
 4180,
 4181,
 4182,
 4183,
 4184,
 4185,
 4186,
 4187,
 61532,
 4188,
 4189,
 4190,
 16480,
 4191,
 4192,
 4193,
 4194,
 4195,
 4196,
 4197,
 4198,
 4199,
 4200,
 4201,
 4202,
 4204,
 4205,
 4206,
 4207,
 4203,
 4209,
 4210,
 4208,
 4212,
 4211,
 4214,
 4213,
 4216,
 4217,
 4215,
 32892,
 4219,
 4218,
 4220,
 4222,
 4221,
 4223,
 4224,
 4225,
 4226,
 4227,
 36997,
 4229,
 4228,
 4232,
 4230,
 4231,
 4235,
 4236,
 4237,
 24723,
 4238,
 4240,
 4241,
 4242,
 4239,
 4245,
 4246,
 4247,
 41116,
 4248,
 4250,
 4251,
 4252,
 4253,
 4249,
 4254,
 4256,
 4255,
 4258,
 4257,
 4260,
 4261,
 4262,
 4263,
 4259,
 4264,
 4266,
 4267,
 4268,
 4265,
 41138,
 4270,
 4269,
 4271,
 53425,
 4272,
 4273,
 4276,
 4277,
 4278,
 41148,
 4280,
 4281,
 4282,
 4283,
 4284,
 4287,
 4288,
 4285,
 4290,
 4291,
 4292,
 53445,
 4293,

In [5]:
# Unir sport y dport en una sola columna de nodos
df_subset['node'] = df_subset[['sport', 'dport']].apply(lambda x: frozenset(x), axis=1)

# Seleccionar las columnas de características
feature_columns = df_subset.columns.difference(['sport', 'dport', 'node', 'snortclasses'])

# Calcular el promedio de las características para cada nodo
node_features = df_subset.groupby('node')[feature_columns].mean().reset_index()

# Mostrar el DataFrame de características de los nodos
node_features


,node,aggdip,aggdipsamedport,aggdipsport,aggnumberofflows,aggsip,aggsipdport,bpp,bps,flowconntime,flowprotocol,load,numbytes,numpackets,rate
0,"(80, 50850)",0.0,0.0,0.0,0.0,0.0,0.0,342.250000,17063.229924,0.240693,0.0,124108.304688,4107.0,12.0,45.701370
1,"(11936, 53)",1.0,1.0,1.0,8.0,3.0,1.0,75.000000,8.514395,17.617224,1.0,0.000000,150.0,2.0,0.056763
2,"(80, 50946)",1.0,1.0,1.0,10.0,1.0,1.0,129.272727,108.093094,13.155327,0.0,778.391907,1422.0,11.0,0.760148
3,"(53, 24285)",2.0,2.0,1.0,10.0,5.0,1.0,75.000000,0.000000,0.000000,1.0,0.000000,75.0,1.0,0.000000
4,"(80, 1189)",2.0,2.0,1.0,9.0,2.0,1.0,124.800000,1828.057162,0.682692,0.0,13007.330078,1248.0,10.0,13.183105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1314,"(80, 1123)",5.0,5.0,1.0,6.0,5.0,1.0,814.672566,23853.752539,3.859267,0.0,189128.140625,92058.0,113.0,29.021055
1315,"(80, 1122)",4.0,4.0,1.0,5.0,4.0,1.0,815.631579,7373.479663,10.508607,0.0,58360.542969,77485.0,95.0,8.945049
1316,"(80, 1121)",3.0,3.0,1.0,4.0,3.0,1.0,124.800000,1762.241436,0.708189,0.0,12539.025391,1248.0,10.0,12.708472
1317,"(80, 1119)",2.0,2.0,1.0,5.0,2.0,1.0,124.800000,1836.268960,0.679639,0.0,13065.760742,1248.0,10.0,13.242325


In [32]:
concatAll

,origin,destination,weight
0,50850,80,4107
1,50851,80,8802
2,143,1090,1010
3,50852,80,3425
4,2497,80,120
...,...,...,...
2937,80,4391,4645
2938,80,4392,7003
2939,80,4393,6017
2940,80,4394,4919
